### Configuração do LDA

In [ ]:
import pandas as pd
from gensim import corpora
import re

# Carrega o dataset
df_novo = pd.read_csv('./datasets/database-lemmetizado.csv.zip', compression='zip')
df_novo = df_novo.sort_values('date_published').reset_index(drop=True)

# Converter a coluna de datas para datetime
df_novo['date_published'] = pd.to_datetime(df_novo['date_published'], errors='coerce')

# Usar esse caso o dataset esteja pre-processado
def tokenize_text(text):
    """
    Função para tokenizar o texto.
    """
    # Converte para minúsculas e separa em palavras
    tokens = text.lower().split()
    return tokens

documents = df_novo['tokens'].dropna().astype(str).tolist()
processed_docs = [tokenize_text(doc) for doc in documents]

# Cria dicionário e corpus
dictionary = corpora.Dictionary(processed_docs)
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]

print(f"Número de documentos: {len(corpus)}")
print(f"Tamanho do dicionário: {len(dictionary)}")
print(f"Exemplo de documento (Bag-of-Words): {corpus[0]}")
print(f"processed_docs[0]: {processed_docs[0]}")

['1999-1999', '2000-2000', '2001-2001', '2002-2002', '2003-2003', '2004-2004', '2005-2005', '2006-2006', '2007-2007', '2008-2008', '2009-2009', '2010-2010', '2011-2011', '2012-2012', '2013-2013', '2014-2014', '2015-2015', '2016-2016', '2017-2017', '2018-2018', '2019-2019', '2020-2020', '2021-2021', '2022-2022', '2023-2023', '2024-2024']
Número de documentos: 25240
Tamanho do dicionário: 47851
Exemplo de documento (Bag-of-Words): [(0, 1), (1, 1), (2, 1), (3, 4), (4, 2), (5, 2), (6, 1), (7, 4), (8, 1), (9, 3), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (15, 1), (16, 1), (17, 1), (18, 3), (19, 1), (20, 1), (21, 1), (22, 1), (23, 1), (24, 1), (25, 1), (26, 1), (27, 1), (28, 1), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 4), (35, 1), (36, 1)]
processed_docs[0]: ["['dispositivo',", "'eletrocrèmico',", "'combinação',", "'eletrocrèmica',", "'patente',", "'invenção',", "'dispositivo',", "'eletrocrèmico',", "'combinação',", "'eletrocrèmico',", "'apresentar',", "'dispositivo',", "'eletro

### Treinamento LDA

In [ ]:
from gensim import corpora, models
import os

# --- FUNÇÃO PRINCIPAL FOCADA EM TREINAR E SALVAR ---
def train_and_save_lda_per_year(df, num_topics=10, passes=10, alpha='auto', eta='auto'):
    """
    Treina um modelo LDA para cada ano e salva-o individualmente no disco.
    A função foca apenas na execução, sem retornar estatísticas.

    Args:
        df (pd.DataFrame): DataFrame com 'date_published' (datetime) e 'tokens'.
        num_topics (int): Número de tópicos para cada modelo LDA.
        passes (int): Número de passes de treinamento.
        alpha (str or float): Parâmetro alpha do LDA.
        eta (str or float): Parâmetro eta do LDA.
    """
    
    # Extrai o ano das datas
    df['year'] = df['date_published'].dt.year
    years = sorted(df['year'].dropna().unique())
    
    print(f"Anos encontrados no dataset: {years}")
    
    # Define o diretório onde os modelos serão salvos
    output_dir = './modelos/lda_por_ano'
    os.makedirs(output_dir, exist_ok=True)
    print(f"Modelos serão salvos em: '{output_dir}'")
    
    models_trained_count = 0
    
    for year in years:
        print(f"\n=== Processando ano: {year} ===")
        
        df_year = df[df['year'] == year].copy()
        
        documents = df_year['tokens'].dropna().tolist()
        
        processed_docs = [doc.split() if isinstance(doc, str) else doc for doc in documents]
        
        processed_docs = [doc for doc in processed_docs if doc]
        
        print(f"Documentos para treinamento: {len(processed_docs)}")
        
        dictionary_year = corpora.Dictionary(processed_docs)
        corpus_year = [dictionary_year.doc2bow(doc) for doc in processed_docs]
        
        try:
            # Treina o modelo LDA
            lda_model = models.LdaModel(
                corpus=corpus_year,
                id2word=dictionary_year,
                num_topics=num_topics,
                random_state=42,
                passes=passes,
                alpha=alpha,
                eta=eta
            )
            
            # Salva o modelo treinado
            model_path = os.path.join(output_dir, f'lda_model_{year}.model')
            lda_model.save(model_path)
            
            print(f"Modelo para o ano {year} treinado e salvo com sucesso em '{model_path}'")
            models_trained_count += 1
            
        except Exception as e:
            print(f"ERRO ao treinar o modelo para o ano {year}: {e}")
            continue
    
    print("\n=== Treinamento Concluído ===")
    print(f"Total de modelos treinados e salvos: {models_trained_count}")

print("Iniciando o processo de treinamento de modelos LDA por ano...")
train_and_save_lda_per_year(
    df=df_novo, 
    num_topics=10,
    passes=10
)

print("\nProcesso finalizado.")

Iniciando o processo de treinamento de modelos LDA por ano...
Anos encontrados no dataset: [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Modelos serão salvos em: './modelos/lda_por_ano'

=== Processando ano: 1999 ===
Documentos para treinamento: 223
Modelo para o ano 1999 treinado e salvo com sucesso em './modelos/lda_por_ano\lda_model_1999.model'

=== Processando ano: 2000 ===
Documentos para treinamento: 1166
Modelo para o ano 2000 treinado e salvo com sucesso em './modelos/lda_por_ano\lda_model_2000.model'

=== Processando ano: 2001 ===
Documentos para treinamento: 1084
Modelo para o ano 2001 treinado e salvo com sucesso em './modelos/lda_por_ano\lda_model_2001.model'

=== Processando ano: 2002 ===
Documentos para treinamento: 899
Modelo para o ano 2002 treinado e salvo com sucesso em './modelos/lda_por_ano\lda_model_2002.model'

=== Processando ano: 2003 ===
Documentos par

### Gráficos

In [ ]:
from gensim.models import LdaModel

def display_topics_from_models_with_probs(model_dir, num_words=10):
    """
    Carrega modelos LDA de um diretório e exibe os tópicos de cada um,
    incluindo a probabilidade de cada palavra formatada como percentagem.

    Args:
        model_dir (str): O caminho para o diretório onde os modelos .model estão salvos.
        num_words (int): O número de palavras a serem exibidas para cada tópico.
    """
    
    if not os.path.exists(model_dir):
        print(f"ERRO: O diretório '{model_dir}' não foi encontrado.")
        return

    try:
        model_files = [f for f in os.listdir(model_dir) if f.startswith('lda_model_') and f.endswith('.model')]
    except FileNotFoundError:
        print(f"ERRO: O diretório '{model_dir}' não foi encontrado.")
        return

    if not model_files:
        print(f"Nenhum ficheiro de modelo (.model) encontrado em '{model_dir}'.")
        return
        
    model_files.sort()
    
    print(f"Encontrados {len(model_files)} modelos. Exibindo os tópicos para cada ano...")
    
    for filename in model_files:
        try:
            match = re.search(r'_(\d{4})\.model', filename)
            if not match:
                continue
            
            year = match.group(1)
            model_path = os.path.join(model_dir, filename)
            lda_model = LdaModel.load(model_path)
            
            print(f"\n--- Tópicos para o Ano: {year} ---")
            
            topics = lda_model.show_topics(num_topics=-1, num_words=num_words, formatted=False)
            
            for topic_id, word_probs in topics:
                # --- AQUI ESTÁ A MUDANÇA PRINCIPAL ---
                # Para cada par (palavra, probabilidade), criamos uma string formatada "palavra (X.XX%)"
                formatted_words = [f"{word} ({prob*100:.2f}%)" for word, prob in word_probs]
                
                # Junta as palavras já formatadas numa única string para exibição
                print(f"Tópico {topic_id}: {', '.join(formatted_words)}")
                
        except Exception as e:
            print(f"\nERRO ao processar o ficheiro {filename}: {e}")
            continue

MODEL_DIRECTORY = './modelos/lda_por_ano'

WORDS_PER_TOPIC = 8

display_topics_from_models_with_probs(MODEL_DIRECTORY, num_words=WORDS_PER_TOPIC)

Encontrados 26 modelos. Exibindo os tópicos para cada ano...

--- Tópicos para o Ano: 1999 ---
Tópico 0: 'porta', (4.30%), 'elevador', (2.21%), 'motor', (2.11%), 'carro', (1.76%), 'acionamento', (1.46%), 'movimento', (1.18%), 'abertura', (0.87%), 'dispositivo', (0.76%)
Tópico 1: 'fixador', (1.25%), 'elemento', (0.88%), 'remoto', (0.75%), 'material', (0.70%), 'incluir', (0.67%), 'painel', (0.67%), 'retenção', (0.67%), 'articulação', (0.67%)
Tópico 2: 'inferior', (1.28%), 'lateral', (1.01%), 'parede', (0.99%), 'aba', (0.92%), 'peça', (0.79%), 'trilho', (0.68%), 'mola', (0.68%), 'extremidade', (0.65%)
Tópico 3: 'travamento', (0.99%), 'fecho', (0.99%), 'invenção', (0.92%), 'peça', (0.73%), 'outro', (0.73%), 'lâmina', (0.73%), 'direção', (0.67%), 'elemento', (0.65%)
Tópico 4: 'construção', (1.71%), 'vedação', (1.14%), 'parede', (0.99%), 'invenção', (0.90%), 'elemento', (0.77%), 'outro', (0.76%), 'patente', (0.73%), 'caixa', (0.70%)
Tópico 5: 'dobradiça', (2.26%), 'porta', (1.83%), 'pino', (